# 1D s-wave Superconducting Mean-Field SCF

This notebook shows the spin-singlet s-wave attractive-Hubbard SCF path. The normal Hamiltonian is spinful, the BdG Hamiltonian is rebuilt at each iteration from the current onsite pairing and Hartree profiles, and the new fields are obtained from the local normal and anomalous densities.


In [ ]:
using LinearAlgebra
using Plots
using ITensors
using ITensorMPS

include("../src/TensorBinding.jl")
using .TensorBinding

## 1. Spinful normal chain

Onsite s-wave pairing needs spin. We therefore start from a spinless tight-binding chain and add a spin degree of freedom before entering the superconducting SCF loop.

In [ ]:
L = 3
N = 2^L
t = 1.0
mu = 0.0

H_seed = TensorBinding.get_Hamiltonian("chain_1d", t; L=L, scale=4.0)
pos_sites = H_seed.sites

H0 = deepcopy(H_seed)
TensorBinding.add_spin!(H0)

println(H0)


## 2. Initial pairing seed

A uniform positive seed is enough for this minimal s-wave example. For spatially varying seeds, pass a function or an MPS as `initial_delta`.

In [ ]:
Delta0 = 0.15
Delta0_mps = TensorBinding.scf_profile_mps(
    L,
    pos_sites,
    n -> Delta0;
    type=ComplexF64,
    tol=1e-10,
)

Delta0_vals = [TensorBinding.scf_eval_profile_mps(Delta0_mps, n) for n in 0:N-1]

plot(0:N-1, real.(Delta0_vals);
     marker=:circle,
     xlabel="site n",
     ylabel="Delta",
     title="Initial s-wave pairing seed",
     legend=false)


## 3. Pairing + Hartree SCF

The attractive-Hubbard wrapper uses positive `U` as attraction: `Delta_new = -U * F_singlet`, `V_up = -U * (rho_down - background)`, and `V_down = -U * (rho_up - background)`.


In [ ]:
U = 3

result = TensorBinding.scf_swave_hubbard(
    H0,
    U;
    initial_delta=Delta0_mps,
    background=0.5,
    mu=mu,
    density_method=:mcweeny,
    scale=4.8,
    max_scf_iter=35,
    purif_maxiter=30,
    purif_tol=1e-5,
    scf_tol=1e-3,
    mix=0.8,
    maxdim=90,
    cutoff=1e-8,
    verbose=true,
)

println("converged = ", result.converged)
println("iterations = ", result.iterations)
println("final RMS = ", result.rms_error)


## 4. Pairing, density, and convergence


In [ ]:
Delta = [TensorBinding.scf_eval_profile_mps(result.delta_mps, n) for n in 0:N-1]
F = [TensorBinding.scf_eval_profile_mps(result.anomalous_mps, n) for n in 0:N-1]
rho_up = [TensorBinding.scf_eval_profile_mps(result.rho_up_mps, n) for n in 0:N-1]
rho_dn = [TensorBinding.scf_eval_profile_mps(result.rho_dn_mps, n) for n in 0:N-1]
hist_iter = [h.iter for h in result.history]
hist_rms = [h.rms_error for h in result.history]

p1 = plot(0:N-1, real.(Delta);
          marker=:circle,
          xlabel="site n",
          ylabel="Re Delta",
          title="Self-consistent s-wave pairing",
          label="Re Delta")
plot!(p1, 0:N-1, abs.(Delta); marker=:diamond, label="abs Delta")

p2 = plot(0:N-1, real.(F);
          marker=:circle,
          xlabel="site n",
          ylabel="Re F_singlet",
          title="Local singlet anomalous profile",
          legend=false)

p3 = plot(0:N-1, real.(rho_up);
          marker=:circle,
          xlabel="site n",
          ylabel="density",
          title="Self-consistent normal densities",
          label="rho up")
plot!(p3, 0:N-1, real.(rho_dn); marker=:diamond, label="rho down")

p4 = plot(hist_iter, hist_rms;
          yscale=:log10,
          marker=:circle,
          xlabel="SCF iteration",
          ylabel="RMS error",
          title="s-wave SCF convergence",
          legend=false)

plot(p1, p2, p3, p4; layout=(4, 1), size=(760, 1080))
